In [1]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

In [19]:
def compress_image(image_path, n_components=10, save_path=None):
    """
    Compress an image using PCA (eigenvalue decomposition)
    """
    # Load image
    img = Image.open(image_path)
    img_array = np.array(img)
    
    # Work with each color channel separately
    compressed = np.zeros_like(img_array)

    all_eigvals = []
    
    for channel in range(3):
        data = img_array[:, :, channel].astype(float)
        h, w = data.shape
        
        # center data
        mean = np.mean(data, axis=0)
        centered = data - mean
        
        # compute eigenvectors
        cov = np.cov(centered.T)
        eigvals, eigvecs = np.linalg.eig(cov)

        eigvals = np.real(eigvals)
        eigvecs = np.real(eigvecs)        
        
        # sort eigenvalues
        idx = np.argsort(eigvals)[::-1]
        eigvals = eigvals[idx]
        eigvecs = eigvecs[:, idx]

        all_eigvals.append(eigvals)
        
        # project and reconstruct
        projection = centered @ eigvecs[:, :n_components]
        reconstructed = (projection @ eigvecs[:, :n_components].T) + mean
        compressed[:, :, channel] = np.clip(reconstructed, 0, 255)

        # compression statistics
        original_size = h * w
        compressed_size = (h * n_components) + (w * n_components) + w
        compression_ratio = (1 - compressed_size / original_size) * 100
        total_variance = np.sum(all_eigvals[0])
        variance_preserved = np.sum(all_eigvals[0][:n_components]) / total_variance * 100
        original_bytes = original_size * 3  # 3 bytes per pixel (RGB)
        compressed_bytes = compressed_size * 3
        bytes_saved = original_bytes - compressed_bytes
        mb_saved = bytes_saved / (1024 * 1024)        
    
    # Save compressed image
    if save_path:
        Image.fromarray(compressed.astype(np.uint8)).save(save_path)
        print(f'✅ Compressed image saved to: {save_path}')

    return {
        'compression_ratio': round(compression_ratio, 2),
        'variance_preserved': variance_preserved,
        'original_size': original_size,
        'compressed_size': compressed_size,
        'n_components': n_components
    }

In [21]:
compress_image('../img/2026-06-06-01.jpeg', n_components=40, save_path='022_image_compressed.jpg')

✅ Compressed image saved to: 022_image_compressed.jpg


{'compression_ratio': 63.05,
 'variance_preserved': np.float64(98.9296037879935),
 'original_size': 50325,
 'compressed_size': 18595,
 'n_components': 40}